##  Import Core libraries :

In [1]:
import warnings

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

print("Preprocessing tools imported successfully.")
# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Ignore unnecessary warnings for cleaner output
warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_STATE = 42

print("Libraries imported successfully.")
print("Random state:", RANDOM_STATE)

Preprocessing tools imported successfully.
Libraries imported successfully.
Random state: 42


In [2]:
train = pd.read_csv("train (2).csv")
test = pd.read_csv("test (2).csv")
sample_submission = pd.read_csv("gender_submission.csv")
print("Data loaded successfully.")
print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")

Data loaded successfully.
Train shape: (891, 12)
Test shape : (418, 11)


In [3]:
# CREATE WORKING COPIES
train_df = train.copy()
test_df = test.copy()

print("Working copies created.")

Working copies created.


In [4]:
print("TRAIN COLUMN INFORMATION")
train_df.info()

print("\n" + "=" * 70)

print("TEST COLUMN INFORMATION")
test_df.info()

TRAIN COLUMN INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

TEST COLUMN INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   

In [5]:
# MISSING VALUES
print("TRAIN MISSING VALUES")
train_missing = pd.DataFrame({
    "Missing_Count": train_df.isnull().sum(),
    "Missing_Percentage": train_df.isnull().mean() * 100
}).sort_values("Missing_Count", ascending=False)

display(train_missing)

print("\nTEST MISSING VALUES")
test_missing = pd.DataFrame({
    "Missing_Count": test_df.isnull().sum(),
    "Missing_Percentage": test_df.isnull().mean() * 100
}).sort_values("Missing_Count", ascending=False)

display(test_missing)

TRAIN MISSING VALUES


,Missing_Count,Missing_Percentage
Cabin,687,77.104377
Age,177,19.865320
Embarked,2,0.224467
PassengerId,0,0.000000
Name,0,0.000000
Pclass,0,0.000000
Survived,0,0.000000
Sex,0,0.000000
Parch,0,0.000000
SibSp,0,0.000000



TEST MISSING VALUES


,Missing_Count,Missing_Percentage
Cabin,327,78.229665
Age,86,20.574163
Fare,1,0.239234
Name,0,0.000000
Pclass,0,0.000000
PassengerId,0,0.000000
Sex,0,0.000000
Parch,0,0.000000
SibSp,0,0.000000
Ticket,0,0.000000


In [6]:
# DUPLICATE CHECK

print("Duplicate rows in train:", train_df.duplicated().sum())
print("Duplicate rows in test :", test_df.duplicated().sum())

Duplicate rows in train: 0
Duplicate rows in test : 0


In [7]:
# UNIQUE VALUES

categorical_candidates = ["Sex", "Embarked", "Pclass"]

for col in categorical_candidates:
    if col in train_df.columns:
        print(f"\n{col} unique values:")
        print(train_df[col].unique())


Sex unique values:
['male' 'female']

Embarked unique values:
['S' 'C' 'Q' nan]

Pclass unique values:
[3 1 2]


In [10]:
display(train_df.describe().T)

,count,mean,std,min,25%,50%,75%,max
PassengerId,891.0,446.000000,257.353842,1.00,223.5000,446.0000,668.5,891.0000
Survived,891.0,0.383838,0.486592,0.00,0.0000,0.0000,1.0,1.0000
Pclass,891.0,2.308642,0.836071,1.00,2.0000,3.0000,3.0,3.0000
Age,714.0,29.699118,14.526497,0.42,20.1250,28.0000,38.0,80.0000
SibSp,891.0,0.523008,1.102743,0.00,0.0000,0.0000,1.0,8.0000
Parch,891.0,0.381594,0.806057,0.00,0.0000,0.0000,0.0,6.0000
Fare,891.0,32.204208,49.693429,0.00,7.9104,14.4542,31.0,512.3292


In [11]:
# TARGET DISTRIBUTION

target_counts = train_df["Survived"].value_counts().sort_index()
target_percent = train_df["Survived"].value_counts(normalize=True).sort_index() * 100

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage": target_percent.round(2)
})

display(target_summary)

,Count,Percentage
Survived,,
0,549,61.62
1,342,38.38


In [12]:
# FINAL SANITY CHECK: confirms that the expected relationship: Survived should exist only in training data.

print("Train columns:")
print(train_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

print("\nTarget present in train:", "Survived" in train_df.columns)
print("Target present in test :", "Survived" in test_df.columns)

print("\nPassengerId overlap check:")
print(
    "Common PassengerId values:",
    len(set(train_df["PassengerId"]).intersection(set(test_df["PassengerId"]))))

Train columns:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Test columns:
['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Target present in train: True
Target present in test : False

PassengerId overlap check:
Common PassengerId values: 0


### MISSING VALUE TREATMENT & FEATURE ENGINEERING

In [13]:
train_fe = train_df.copy()
test_fe = test_df.copy()

# Keep target separate
y = train_fe["Survived"].copy()
# Combine only for consistent FEATURE CREATION.
# We will not use Survived from test (it doesn't exist).
combined = pd.concat(
    [train_fe.drop(columns=["Survived"]), test_fe],
    axis=0,
    ignore_index=True)
print("Combined shape:", combined.shape)

Combined shape: (1309, 11)


In [14]:
# EXTRACT TITLE

combined["Title"] = combined["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
print(combined["Title"].value_counts())

# Group uncommon titles into "Rare"
title_counts = combined["Title"].value_counts()
rare_titles = title_counts[title_counts < 10].index
combined["Title"] = combined["Title"].replace(rare_titles, "Rare")

# Standardize some equivalent titles
combined["Title"] = combined["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"})
print(combined["Title"].value_counts())

Title
Mr              757
Miss            260
Mrs             197
Master           61
Rev               8
Dr                8
Col               4
Major             2
Mlle              2
Ms                2
Mme               1
Don               1
Sir               1
Lady              1
Capt              1
the Countess      1
Jonkheer          1
Dona              1
Name: count, dtype: int64
Title
Mr        757
Miss      260
Mrs       197
Master     61
Rare       34
Name: count, dtype: int64


In [15]:
print(combined.columns.tolist())
# FAMILY SIZE
combined["FamilySize"] = combined["SibSp"] + combined["Parch"] + 1

print(combined["FamilySize"].head())

['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'Title']
0    2
1    2
2    1
3    2
4    1
Name: FamilySize, dtype: int64


In [16]:
# IS ALONE

combined["IsAlone"] = (combined["FamilySize"] == 1).astype(int)
print(combined["IsAlone"].value_counts())

IsAlone
1    790
0    519
Name: count, dtype: int64


In [17]:
# CABIN DECK
combined["Deck"] = combined["Cabin"].str[0]

# Treat missing cabins explicitly
combined["Deck"] = combined["Deck"].fillna("U")
print(combined["Deck"].value_counts())

Deck
U    1014
C      94
B      65
D      46
E      41
A      22
F      21
G       5
T       1
Name: count, dtype: int64


here U means Unknown.

## Handling values

In [18]:
print("Missing Embarked:", combined["Embarked"].isna().sum())

print("\nEmbarked distribution:")
print(combined["Embarked"].value_counts(dropna=False))

Missing Embarked: 2

Embarked distribution:
Embarked
S      914
C      270
Q      123
NaN      2
Name: count, dtype: int64


In [19]:
# Fill missing Embarked with the most common category
embarked_mode = combined["Embarked"].mode()[0]

combined["Embarked"] = combined["Embarked"].fillna(embarked_mode)

print("Missing Embarked after filling:", combined["Embarked"].isna().sum())
#Embarked has only a very small number of missing observations,
#so replacing them with most common category is simple,unlikely to distort dataset significantly.

Missing Embarked after filling: 0


In [20]:
# FARE — MISSING VALUES
print("Missing Fare before:", combined["Fare"].isna().sum())

combined["Fare"] = combined.groupby("Pclass")["Fare"].transform(
    lambda x: x.fillna(x.median()))
print("Missing Fare after :", combined["Fare"].isna().sum())

Missing Fare before: 1
Missing Fare after : 0


In [21]:
combined["TicketGroupSize"] = combined.groupby("Ticket")["Ticket"].transform("count")

In [22]:
combined["FarePerPerson"] = (
    combined["Fare"] / combined["TicketGroupSize"])
#Fare differs substantially by Pclass, 
#so class-specific median imputation is more realistic than using one overall median

In [23]:
# Median age for similar passengers
age_group_median = combined.groupby(
    ["Title", "Pclass", "Sex"]
)["Age"].transform("median")

combined["Age"] = combined["Age"].fillna(age_group_median)

# Safety fallback in case any values still remain
combined["Age"] = combined["Age"].fillna(combined["Age"].median())

print("Missing Age after:", combined["Age"].isna().sum())
# Age varies systematically with title, class, and sex,
#so group-based median imputation preserves more realisticage patterns
# than filling every missing value with one global median.

Missing Age after: 0


In [24]:
# AGE BIN

age_bins = [-np.inf, 5, 12, 18, 30, 45, 60, np.inf]
age_labels = [
    "Child",
    "YoungChild",
    "Teen",
    "YoungAdult",
    "Adult",
    "MatureAdult",
    "Senior"]
combined["AgeBin"] = pd.cut(
    combined["Age"],
    bins=age_bins,
    labels=age_labels)
print(combined["AgeBin"].value_counts().sort_index())

AgeBin
Child           56
YoungChild      46
Teen           146
YoungAdult     569
Adult          336
MatureAdult    123
Senior          33
Name: count, dtype: int64


In [25]:
## Create Fare bin
combined["FareBin"] = pd.qcut(
    combined["Fare"],
    q=4,
    labels=["Low", "Medium", "High", "VeryHigh"],
    duplicates="drop"
)

print(combined["FareBin"].value_counts().sort_index())

FareBin
Low         337
Medium      321
High        328
VeryHigh    323
Name: count, dtype: int64


In [26]:
# ============================================================
# CREATE FAMILY SIZE
# ============================================================

combined["FamilySize"] = combined["SibSp"] + combined["Parch"] + 1

print("FamilySize created successfully.")
print(combined["FamilySize"].head())

FamilySize created successfully.
0    2
1    2
2    1
3    2
4    1
Name: FamilySize, dtype: int64


In [29]:
# FAMILY SIZE GROUP
def family_size_category(size):
    if size == 1:
        return "Alone"
    elif size <= 4:
        return "Small"
    elif size <= 7:
        return "Medium"
    else:
        return "Large"
combined["FamilySizeGroup"] = combined["FamilySize"].apply(
    family_size_category)
print(combined["FamilySizeGroup"].value_counts())

FamilySizeGroup
Alone     790
Small     437
Medium     63
Large      19
Name: count, dtype: int64


Survival may not change linearly with family size; a small family can behave differently from a very large travelling group.

## Clenning columns :

In [32]:
drop_columns = [
    "PassengerId",
    "Name",
    "Cabin",
    "Ticket",
    "Surname"]
combined = combined.drop(columns=drop_columns,errors="ignore")
print("Remaining columns:")
print(combined.columns.tolist())
#Keeping raw identifiers and high-cardinality text fields can add noise.

Remaining columns:
['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone', 'Deck', 'TicketGroupSize', 'FarePerPerson', 'AgeBin', 'FareBin', 'FamilySizeGroup']


## FINAL MISSING VALUE CHECK

In [33]:
missing_summary = pd.DataFrame({
    "Missing_Count": combined.isnull().sum(),
    "Missing_Percentage": combined.isnull().mean() * 100
}).sort_values("Missing_Count", ascending=False)

display(missing_summary)

,Missing_Count,Missing_Percentage
Pclass,0,0.0
Sex,0,0.0
Age,0,0.0
SibSp,0,0.0
Parch,0,0.0
Fare,0,0.0
Embarked,0,0.0
Title,0,0.0
FamilySize,0,0.0
IsAlone,0,0.0


### SPLIT BACK INTO TRAIN / TEST

In [34]:
train_fe = combined.iloc[:len(train_df)].copy()
test_fe = combined.iloc[len(train_df):].copy()

# Restore target
train_fe["Survived"] = y.values

print("Final engineered train shape:", train_fe.shape)
print("Final engineered test shape :", test_fe.shape)

Final engineered train shape: (891, 17)
Final engineered test shape : (418, 16)


In [35]:
print("TRAIN — engineered data")
display(train_fe.head())

print("\nTEST — engineered data")
display(test_fe.head())

print("\nTrain dtypes:")
display(train_fe.dtypes)

TRAIN — engineered data


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,IsAlone,Deck,TicketGroupSize,FarePerPerson,AgeBin,FareBin,FamilySizeGroup,Survived
0,3,male,22.0,1,0,7.2500,S,Mr,2,0,U,1,7.25000,YoungAdult,Low,Small,0
1,1,female,38.0,1,0,71.2833,C,Mrs,2,0,C,2,35.64165,Adult,VeryHigh,Small,1
2,3,female,26.0,0,0,7.9250,S,Miss,1,1,U,1,7.92500,YoungAdult,Medium,Alone,1
3,1,female,35.0,1,0,53.1000,S,Mrs,2,0,C,2,26.55000,Adult,VeryHigh,Small,1
4,3,male,35.0,0,0,8.0500,S,Mr,1,1,U,1,8.05000,Adult,Medium,Alone,0



TEST — engineered data


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,IsAlone,Deck,TicketGroupSize,FarePerPerson,AgeBin,FareBin,FamilySizeGroup
891,3,male,34.5,0,0,7.8292,Q,Mr,1,1,U,1,7.82920,Adult,Low,Alone
892,3,female,47.0,1,0,7.0000,S,Mrs,2,0,U,1,7.00000,MatureAdult,Low,Small
893,2,male,62.0,0,0,9.6875,Q,Mr,1,1,U,1,9.68750,Senior,Medium,Alone
894,3,male,27.0,0,0,8.6625,S,Mr,1,1,U,1,8.66250,YoungAdult,Medium,Alone
895,3,female,22.0,1,1,12.2875,S,Mrs,3,0,U,2,6.14375,YoungAdult,Medium,Small



Train dtypes:


Pclass                int64
Sex                  object
Age                 float64
SibSp                 int64
Parch                 int64
Fare                float64
Embarked             object
Title                object
FamilySize            int64
IsAlone               int64
Deck                 object
TicketGroupSize       int64
FarePerPerson       float64
AgeBin             category
FareBin            category
FamilySizeGroup      object
Survived              int64
dtype: object

In [36]:
# Separate target from training data
X = train_fe.drop(columns=["Survived"])
y = train_fe["Survived"].copy()

# Test features
X_test = test_fe.copy()

print("X shape     :", X.shape)
print("y shape     :", y.shape)
print("X_test shape:", X_test.shape)

X shape     : (891, 16)
y shape     : (891,)
X_test shape: (418, 16)


In [37]:
categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['Sex', 'Embarked', 'Title', 'Deck', 'AgeBin', 'FareBin', 'FamilySizeGroup']

Numerical features:
['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'TicketGroupSize', 'FarePerPerson']


In [38]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler())])

In [39]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        ("encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="if_binary"))])

In [40]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num",numeric_pipeline,numerical_features),
        ("cat",categorical_pipeline,categorical_features)],
    remainder="drop")
print("Preprocessor created successfully.")

Preprocessor created successfully.


## TEST PREPROCESSING

In [41]:
X_processed = preprocessor.fit_transform(X)

print("Original feature shape :", X.shape)
print("Processed feature shape:", X_processed.shape)

Original feature shape : (891, 16)
Processed feature shape: (891, 42)


In [42]:
print("Processed data type:", type(X_processed))

if hasattr(X_processed, "toarray"):
    print("Output is a sparse matrix.")
else:
    print("Output is a dense matrix.")

Processed data type: <class 'numpy.ndarray'>
Output is a dense matrix.


In [43]:
# TRANSFORMED FEATURE NAMES
feature_names = preprocessor.get_feature_names_out()

print("Number of transformed features:", len(feature_names))

print("\nFirst 30 transformed features:")
print(feature_names[:30])

Number of transformed features: 42

First 30 transformed features:
['num__Pclass' 'num__Age' 'num__SibSp' 'num__Parch' 'num__Fare'
 'num__FamilySize' 'num__IsAlone' 'num__TicketGroupSize'
 'num__FarePerPerson' 'cat__Sex_male' 'cat__Embarked_C' 'cat__Embarked_Q'
 'cat__Embarked_S' 'cat__Title_Master' 'cat__Title_Miss' 'cat__Title_Mr'
 'cat__Title_Mrs' 'cat__Title_Rare' 'cat__Deck_A' 'cat__Deck_B'
 'cat__Deck_C' 'cat__Deck_D' 'cat__Deck_E' 'cat__Deck_F' 'cat__Deck_G'
 'cat__Deck_T' 'cat__Deck_U' 'cat__AgeBin_Adult' 'cat__AgeBin_Child'
 'cat__AgeBin_MatureAdult']


In [44]:
preprocessor.fit_transform(X)

array([[ 0.82737724, -0.53190045,  0.43279337, ...,  0.        ,
         0.        ,  1.        ],
       [-1.56610693,  0.64927753,  0.43279337, ...,  0.        ,
         0.        ,  1.        ],
       [ 0.82737724, -0.23660596, -0.4745452 , ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.82737724, -0.82719495,  0.43279337, ...,  0.        ,
         0.        ,  1.        ],
       [-1.56610693, -0.23660596, -0.4745452 , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.82737724,  0.20633579, -0.4745452 , ...,  0.        ,
         0.        ,  0.        ]])

In [45]:
# FRESH PREPROCESSOR FOR MODELING
preprocessor = ColumnTransformer(
    transformers=[("num",Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())]),
            numerical_features),("cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            drop="if_binary"))]), categorical_features)], remainder="drop")
print("Fresh preprocessing pipeline is ready for cross-validation.")

Fresh preprocessing pipeline is ready for cross-validation.


In [46]:
print("Missing values in X:")
print(X.isnull().sum().sum())

print("\nMissing values in X_test:")
print(X_test.isnull().sum().sum())

Missing values in X:
0

Missing values in X_test:
0


### MODEL TRAINING & CROSS-VALIDATION

In [47]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
RANDOM_STATE = 42
print("Core modeling libraries imported successfully.")

Core modeling libraries imported successfully.


In [48]:
# XGBoost
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
    print("XGBoost available.")
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost is not installed.")

# LightGBM
try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
    print("LightGBM available.")
except ImportError:
    LGBM_AVAILABLE = False
    print("LightGBM is not installed.")

XGBoost is not installed.
LightGBM is not installed.


**XGBoost and LightGBM are often excellent tabular-data models, 
but availability can differ between Kaggle, Colab, and local Jupyter environments.**

In [49]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

preprocessor_model = ColumnTransformer(
    transformers=[
        ("num",Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())]),
            numerical_features ),("cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        drop="if_binary"))]),categorical_features)],remainder="drop")
print("Fresh model preprocessor created.")

Fresh model preprocessor created.


In [50]:
logistic_model = LogisticRegression(
    C=1.0,
    max_iter=2000,
    solver="liblinear",
    random_state=RANDOM_STATE)

**Logistic Regression is an excellent classification baseline and gives us a useful benchmark against more complex models.**

In [51]:
random_forest_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1)

**Random Forest captures nonlinear relationships and interactions without requiring us to manually specify them**

In [52]:
if XGB_AVAILABLE:

    xgb_model = XGBClassifier(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=2,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1)
else:
    xgb_model = None

**XGBoost builds trees sequentially to correct previous errors, making it especially strong at learning complex patterns in structured/tabular data.**

In [53]:
if LGBM_AVAILABLE:

    lgbm_model = LGBMClassifier(
        n_estimators=400,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=15,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )

else:
    lgbm_model = None

**LightGBM is another efficient gradient-boosting algorithm that can learn nonlinear feature interactions very effectively on tabular datasets.**

In [54]:
# COMPLETE MODEL PIPELINES
models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor_model),
        ("model", logistic_model)]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor_model),
        ("model", random_forest_model)])}
if XGB_AVAILABLE:
    models["XGBoost"] = Pipeline([
        ("preprocessor", preprocessor_model),
        ("model", xgb_model)])
if LGBM_AVAILABLE:
    models["LightGBM"] = Pipeline([
        ("preprocessor", preprocessor_model),
        ("model", lgbm_model)])
print("Models available for comparison:")
for name in models:
    print("-", name)

Models available for comparison:
- Logistic Regression
- Random Forest


**Why : Putting preprocessing and modeling together guarantees that CV never accidentally uses information from the validation fold.**

In [55]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE)
print(cv)

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


In [56]:
scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "roc_auc": "roc_auc"}

In [57]:
# CROSS-VALIDATE ALL MODELS

cv_results = []
for model_name, pipeline in models.items():

    print(f"\nRunning CV for: {model_name}")
    scores = cross_validate(
        estimator=pipeline,
        X=X,
        y=y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False) 
    cv_results.append({
        "Model": model_name,
        "Accuracy Mean": scores["test_accuracy"].mean(),
        "Accuracy Std": scores["test_accuracy"].std(),

        "F1 Mean": scores["test_f1"].mean(),
        "F1 Std": scores["test_f1"].std(),

        "ROC-AUC Mean": scores["test_roc_auc"].mean(),
        "ROC-AUC Std": scores["test_roc_auc"].std()})
# Convert to DataFrame
results_df = pd.DataFrame(cv_results)
# Sort according to competition metric
results_df = results_df.sort_values(
    by="Accuracy Mean",
    ascending=False
).reset_index(drop=True)
display(results_df)


Running CV for: Logistic Regression

Running CV for: Random Forest


,Model,Accuracy Mean,Accuracy Std,F1 Mean,F1 Std,ROC-AUC Mean,ROC-AUC Std
0,Random Forest,0.839502,0.005842,0.782130,0.011601,0.888559,0.017702
1,Logistic Regression,0.827155,0.012073,0.770344,0.021355,0.873124,0.020957


- Accuracy Mean : 0.8300 = model approximately 83% passengers correctly classify kar raha tha.
- Accuracy Std : 0.01 = model relatively stable hai.
- F1 Mean : Precision aur Recall ka balance. Titanic mein supplementary metric hai
- ROC-AUC Mean : Model survivors ko non-survivors se rank karne mein kitna achha hai. 0.5 ≈ random, 1.0 = perfect.

## Cross-validation

In [58]:
display(
    results_df[
        [ "Model",
        "Accuracy Mean",
        "Accuracy Std"]].round(4))

,Model,Accuracy Mean,Accuracy Std
0,Random Forest,0.8395,0.0058
1,Logistic Regression,0.8272,0.0121


In [59]:
best_model_name = results_df.iloc[0]["Model"]

print("Best model based on CV accuracy:")
print(best_model_name)
print(
    "\nCV Accuracy:",
    round(results_df.iloc[0]["Accuracy Mean"], 4))
print(
    "CV Accuracy Std:",
    round(results_df.iloc[0]["Accuracy Std"], 4))

Best model based on CV accuracy:
Random Forest

CV Accuracy: 0.8395
CV Accuracy Std: 0.0058


### HYPERPARAMETER TUNING

In [60]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform

# ============================================================
# PARAMETER SEARCH SPACES
# ============================================================

param_distributions = {}

param_distributions["Logistic Regression"] = {
    "model__C": loguniform(0.01, 10)}
param_distributions["Random Forest"] = {
    "model__n_estimators": randint(300, 800),
    "model__max_depth": [None, 5, 7, 10, 15],
    "model__min_samples_split": randint(2, 10),
    "model__min_samples_leaf": randint(1, 5),
    "model__max_features": ["sqrt", "log2"]}
if XGB_AVAILABLE:
    param_distributions["XGBoost"] = {
        "model__n_estimators": randint(250, 700),
        "model__max_depth": randint(2, 7),
        "model__learning_rate": uniform(0.02, 0.08),
        "model__min_child_weight": randint(1, 6),
        "model__subsample": uniform(0.7, 0.3),
        "model__colsample_bytree": uniform(0.7, 0.3)}
if LGBM_AVAILABLE:
    param_distributions["LightGBM"] = {
        "model__n_estimators": randint(250, 700),
        "model__num_leaves": randint(15, 60),
        "model__learning_rate": uniform(0.02, 0.08),
        "model__min_child_samples": randint(10, 30),
        "model__subsample": uniform(0.7, 0.3),
        "model__colsample_bytree": uniform(0.7, 0.3)}

In [61]:
# TUNE THE BEST MODEL
best_pipeline = models[best_model_name]

search = RandomizedSearchCV(
    estimator=best_pipeline,
    param_distributions=param_distributions[best_model_name],
    n_iter=20,
    scoring="accuracy",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True
)

print(f"Tuning: {best_model_name}")

search.fit(X, y)

print("\nBest CV accuracy:")
print(round(search.best_score_, 4))

print("\nBest parameters:")
print(search.best_params_)

Tuning: Random Forest

Best CV accuracy:
0.8406

Best parameters:
{'model__max_depth': 7, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 4, 'model__min_samples_split': 6, 'model__n_estimators': 399}


In [62]:
# BASELINE VS TUNED

baseline_score = results_df.loc[
    results_df["Model"] == best_model_name,
    "Accuracy Mean"].iloc[0]

tuned_score = search.best_score_
comparison = pd.DataFrame({
    "Version": [
        "Baseline",
        "Tuned"],
    "CV Accuracy": [
        baseline_score,
        tuned_score]})
comparison["CV Accuracy"] = comparison["CV Accuracy"].round(4)

display(comparison)

,Version,CV Accuracy
0,Baseline,0.8395
1,Tuned,0.8406


In [63]:
final_model = search.best_estimator_

print("Final model:")
print(final_model)

Final model:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Pclass', 'Age', 'SibSp',
                                                   'Parch', 'Fare',
                                                   'FamilySize', 'IsAlone',
                                                   'TicketGroupSize',
                                                   'FarePerPerson']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                        

## Final Predictions :

In [64]:
# Generate predictions on the original test features
test_predictions = final_model.predict(X_test)

print("Predictions generated successfully.")
print("Number of predictions:", len(test_predictions))

# Check prediction values
print("\nUnique predicted classes:")
print(np.unique(test_predictions))

print("\nPrediction distribution:")
print(pd.Series(test_predictions).value_counts().sort_index())

Predictions generated successfully.
Number of predictions: 418

Unique predicted classes:
[0 1]

Prediction distribution:
0    266
1    152
Name: count, dtype: int64


### CREATING SUBMISSION DATAFRAME

In [65]:
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": test_predictions.astype(int)})
print("Submission shape:", submission.shape)
display(submission.head(10))

Submission shape: (418, 2)


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0


## SUBMISSION VERIFICATION


In [66]:
print("Submission columns:")
print(submission.columns.tolist())

print("\nExpected columns:")
print(["PassengerId", "Survived"])

print("\nMissing values:")
print(submission.isnull().sum())

print("\nDuplicate PassengerIds:")
print(submission["PassengerId"].duplicated().sum())

print("\nSurvived unique values:")
print(sorted(submission["Survived"].unique()))

Submission columns:
['PassengerId', 'Survived']

Expected columns:
['PassengerId', 'Survived']

Missing values:
PassengerId    0
Survived       0
dtype: int64

Duplicate PassengerIds:
0

Survived unique values:
[np.int64(0), np.int64(1)]


In [67]:
ids_match = submission["PassengerId"].equals(
    test_df["PassengerId"].reset_index(drop=True))

print("Passenger IDs match test.csv:", ids_match)

Passenger IDs match test.csv: True


In [68]:
submission_path = "submission.csv"

submission.to_csv(
    submission_path,
    index=False)
print(f"Submission saved successfully: {submission_path}")

Submission saved successfully: submission.csv


In [69]:
print("File exists:", os.path.exists(submission_path))

if os.path.exists(submission_path):
    print("File size:", os.path.getsize(submission_path), "bytes")

File exists: True
File size: 3258 bytes


In [70]:
submission_check = pd.read_csv(submission_path)

print("Reloaded submission shape:", submission_check.shape)

display(submission_check.head())

print("\nSubmission info:")
submission_check.info()

Reloaded submission shape: (418, 2)


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1



Submission info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   PassengerId  418 non-null    int64
 1   Survived     418 non-null    int64
dtypes: int64(2)
memory usage: 6.7 KB


In [71]:
# ============================================================
# FINAL KAGGLE SUBMISSION CHECKLIST
# ============================================================

checks = {
    "Correct number of rows": len(submission) == len(test_df),
    "Correct columns": list(submission.columns) == ["PassengerId", "Survived"],
    "No missing values": submission.isnull().sum().sum() == 0,
    "No duplicate PassengerIds": submission["PassengerId"].duplicated().sum() == 0,
    "Passenger IDs match test": submission["PassengerId"].equals(
        test_df["PassengerId"].reset_index(drop=True)
    ),
    "Only valid target classes": set(submission["Survived"].unique()).issubset({0, 1}),
    "File exists": os.path.exists(submission_path)
}

print("FINAL CHECKLIST")
print("=" * 50)

all_passed = True

for check, result in checks.items():
    status = "PASS" if result else "FAIL"
    print(f"{status:>5} | {check}")

    if not result:
        all_passed = False

print("=" * 50)

if all_passed:
    print("All checks passed. submission.csv is ready for Kaggle!")
else:
    print("Some checks failed. Review the output above.")

FINAL CHECKLIST
 PASS | Correct number of rows
 PASS | Correct columns
 PASS | No missing values
 PASS | No duplicate PassengerIds
 PASS | Passenger IDs match test
 PASS | Only valid target classes
 PASS | File exists
All checks passed. submission.csv is ready for Kaggle!
